In [1]:
from qiskit import QuantumCircuit
from typing import List
import pandas as pd

class encoding_strategy():
    def generate_pubs(self, data : pd.DataFrame) -> list[tuple[QuantumCircuit, List[float]]]:
        pass

In [2]:
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import random_unitary
import numpy as np
from sklearn.preprocessing import MinMaxScaler

class heisenberg_encoding_strategy(encoding_strategy):
    '''https://arxiv.org/pdf/2509.17715'''
    def __init__(self, n_qubits : int, n_features : int, seed=42, scale=True):
        self.n_qubits = n_qubits
        self.n_features = n_features
        self.scale = scale
        rng = np.random.default_rng(seed=seed)
        self.unitaries = [random_unitary(2, seed=rng.integers(0, 2**32)) for _ in range(n_qubits)]

    def generate_pubs(self, data : pd.DataFrame) -> list[tuple[QuantumCircuit, List[float]]]:
        parameters = ParameterVector('x', self.n_features)
        pubs = []

        qc = QuantumCircuit(self.n_qubits)
        for i in range(self.n_qubits):
            qc.append(self.unitaries[i], [i])

        for num_encoded_features in range(self.n_features):
            start_qubit = num_encoded_features % (self.n_qubits - 1)
            qc.rzz(parameters[num_encoded_features], start_qubit, start_qubit + 1)
            qc.ryy(parameters[num_encoded_features], start_qubit, start_qubit + 1)
            qc.rxx(parameters[num_encoded_features], start_qubit, start_qubit +1)

        # Scale input data into the interval [0, 1].
        if self.scale:
            scaled_data = MinMaxScaler().fit_transform(data.values)
        else:
            scaled_data = np.asarray(data.values, dtype=float)

        # Rescale normalized values into angular encoding range [0, 2π].
        scaled_data = 2 * np.pi * scaled_data
        n_params = self.n_features
        # Zero-pad feature vectors if there are fewer classical features
        # than circuit parameters.
        if scaled_data.shape[1] < n_params:
            scaled_data = np.hstack([
                scaled_data,
                np.zeros((
                    scaled_data.shape[0],
                    n_params - scaled_data.shape[1]
                ))
            ])
        pubs.append((qc, row.tolist()))
        return pubs